# Example of deriving snow depths, on a map, echogram and uncertainties

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.append(os.path.abspath('../Pathfinder'))

from helper_functions import *
from radar_functions import *
from pathfinder import *
from attach_grounddata import *
from add_dataflashlog import *
from clicki_tool import *

%matplotlib inline

In [ ]:
campaignID = '2024_ESASnowDrone'

RADARs = {}
for datasetID in ['26_04_2024_3', '26_04_2024_4', '26_04_2024_5']:
    RADAR = load_RADAR(radar_type='UWiBaSS',
                    datasetID=datasetID,
                    campaignID=campaignID,
                    yaml_file="./campaigns.yaml"
                    )
    zone_number = utm.latlon_to_zone_number(np.nanmedian(RADAR.GPS_Lat), np.nanmedian(RADAR.GPS_Lng))
    RADAR.UTM_zone = zone_number
    UTM_transformer = Transformer.from_crs(4326, CRS.from_proj4(f"+proj=utm +zone={RADAR.UTM_zone} +ellps=WGS84 +datum=WGS84 +units=m +type=crs"), always_xy=True)
    RADAR, df_MP, df_SP, dict_SP = add_insitu(RADAR, UTM_transformer)

    RADARs[datasetID] = RADAR
    
    

In [ ]:
# Example: resample each dataset's snow depth to regular spacing
resampled_snow_depths = {}

for key, uwi in RADARs.items():
    
    dist = np.array(uwi.dist)
    regular_spacing = np.mean(np.diff(dist))  # meters
    
    snow_depth = np.array(uwi.PF_snow_depth)
    mask = np.isfinite(dist) & np.isfinite(snow_depth)
    dist = dist[mask]
    snow_depth = snow_depth[mask]

    if len(dist) > 1:
        dist_regular = np.arange(dist.min(), dist.max(), regular_spacing)
        interp_func = interp1d(dist, snow_depth, kind='linear', fill_value='extrapolate')
        resampled_snow_depths[key] = interp_func(dist_regular)
    else:
        resampled_snow_depths[key] = np.array([])  # Not enough points to interpolate

In [ ]:
all_GPS_Statii = [uwi.GPS_Status for uwi in RADARs.values()]
all_GPS_Statii = np.concatenate(all_GPS_Statii)

all_idx_from_top = [uwi.PF_internal_layers['idx_from_top'] for uwi in RADARs.values()]
all_idx_from_top = np.concatenate(all_idx_from_top)

all_idx_from_bottom = [uwi.PF_internal_layers['idx_from_bottom'] for uwi in RADARs.values()]
all_idx_from_bottom = np.concatenate(all_idx_from_bottom)

# all_value_at_internal  = [uwi.rx_rpca[np.array(uwi.PF_internal_layers['idx_from_top']) + np.array(uwi.PF_top_interface), range(len(uwi.PF_internal_layers['idx_from_top']))] for uwi in data_dict.values()]

all_internal_SNR = [uwi.PF_internal_layers['SNR'] for uwi in RADARs.values()]
all_internal_SNR = np.concatenate(all_internal_SNR)

all_internal_UTM_x = []
all_internal_UTM_y = []

all_N_layers = []
all_value_at_internal  = []

for uwi in RADARs.values():
    internal_x = [list(zip(*p))[1] for p in uwi.PF_internal_layers['paths']] 
    internal_y = [list(zip(*p))[0] for p in uwi.PF_internal_layers['paths']]
    internal_x = np.concatenate(internal_x)
    internal_y = np.concatenate(internal_y)
    try:
        all_value_at_internal.append(uwi.rx_rpca[internal_y, internal_x])
    except:
        all_value_at_internal.append(uwi.rx_rpca_T[internal_y, internal_x])
    
    N_layers = [len(internal_y[internal_x == x]) - 1 for x in internal_x]
    all_N_layers.append(N_layers)
    all_internal_UTM_x.append(uwi.log_UTM_x[internal_x])
    all_internal_UTM_y.append(uwi.log_UTM_y[internal_x])

all_value_at_internal = np.concatenate(all_value_at_internal)
all_N_layers = np.concatenate(all_N_layers)
all_internal_UTM_x = np.concatenate(all_internal_UTM_x)
all_internal_UTM_y = np.concatenate(all_internal_UTM_y)

all_snow_depths_indices = [uwi.PF_bottom_interface - uwi.PF_top_interface for uwi in RADARs.values()]
all_snow_depths_indices = np.concatenate(all_snow_depths_indices)

all_snow_depths_resampled = [sd for sd in resampled_snow_depths.values()]
all_snow_depths_resampled = np.concatenate(all_snow_depths_resampled)

all_snow_depths = [uwi.PF_snow_depth for uwi in RADARs.values()]
all_snow_depths = np.concatenate(all_snow_depths)

all_UTM_x = [uwi.log_UTM_x for uwi in RADARs.values()]
all_UTM_y = [uwi.log_UTM_y for uwi in RADARs.values()]
all_UTM_y = np.concatenate(all_UTM_y)
all_UTM_x = np.concatenate(all_UTM_x)

all_lon = [uwi.GPS_Lng for uwi in RADARs.values()]
all_lat = [uwi.GPS_Lat for uwi in RADARs.values()]
all_lat = np.concatenate(all_lat)
all_lon = np.concatenate(all_lon)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': ccrs.UTM(33)}, constrained_layout=True, dpi=150)


for uwi in data_dict.values():
    mask = uwi.altitude_mask
    scat = ax.scatter(uwi.log_UTM_x[mask], uwi.log_UTM_y[mask], c=uwi.PF_snow_depth[mask],
               norm=TwoSlopeNorm(vmin=np.quantile(all_snow_depths, 0.01), vcenter=np.median(all_snow_depths), vmax=np.quantile(all_snow_depths, 0.99)),
               cmap=cmr.freeze,
               s=2,
               zorder=3,
               transform=ccrs.UTM(uwi.utm_zone),
               )
    ax.scatter(uwi.log_UTM_x[mask], uwi.log_UTM_y[mask],
               s=20,
               zorder=1,
               alpha=.1,
               transform=ccrs.UTM(uwi.utm_zone),
               label=uwi.dataset_name
               )
    
ax.scatter(df_SP['UTM_x'], df_SP['UTM_y'],
           fc='goldenrod', ec='black',
           s=150,
           marker='*',
           transform=ccrs.UTM(uwi.utm_zone),
           label='Snow pit',
           zorder=10000
           
           )
ins_ax = ax.inset_axes([0.01, 0.01, 0.18, 0.32])
ins_ax.set_xticks([])
ins_ax.set_yticks([])
ins_ax.spines['top'].set_visible(False)
ins_ax.spines['right'].set_visible(False)
ins_ax.spines['left'].set_visible(False)
ins_ax.spines['bottom'].set_visible(False)
ins_ax.set_facecolor((204/255, 204/255, 204/255, 0.4))

ax.add_wmts(url, layer, alpha=.8)

ins_ax = ax.inset_axes([0.79, 0.01, 0.2, 0.16])
ins_ax.set_xticks([])
ins_ax.set_yticks([])
ins_ax.spines['top'].set_visible(False)
ins_ax.spines['right'].set_visible(False)
ins_ax.spines['left'].set_visible(False)
ins_ax.spines['bottom'].set_visible(False)
ins_ax.set_facecolor((204/255, 204/255, 204/255, 0.4))

inset_ax = ax.inset_axes([0.8, 0.11, 0.18, 0.04])
cbar = fig.colorbar(
    scat, cax=inset_ax, orientation='horizontal', label='Snow depth [m]',# labelcolor='white',
    extend='both',
    ticks=[0.5, 1, 1.5, 2]
)
cbar.outline.set_visible(False)
# cbar.solids.set_edgecolor("face")
cbar.solids.set_rasterized(True)
# cbar.ax.xaxis.set_tick_params(labelcolor='white')  
xlims = ax.get_xlim()
ylims = ax.get_ylim()

ax.scatter(df_MP['UTM_x'], df_MP['UTM_y'],
           c='red',
           s=10,
           marker='v',
           transform=ccrs.UTM(uwi.utm_zone),
           label='Magnaprobe',
           zorder=100
          )

custom_legend_elements = [
    Line2D([0], [0], color='pink', alpha=.7, lw=3, solid_capstyle='round', label='River outline'),
    
    Line2D([0], [0], color='C0', lw=5, solid_capstyle='round', label='26_04_2024_3'),
    Line2D([0], [0], color='C1', lw=5, solid_capstyle='round', label='26_04_2024_4'),
    Line2D([0], [0], color='C2', lw=5, solid_capstyle='round', label='26_04_2024_5'),
    Line2D([0], [0], color='limegreen', alpha=.5, lw=7, solid_capstyle='round', label='Panels B&C subset'),
    
    Line2D([0], [0], marker='*', color='none', markeredgecolor='black', markerfacecolor='goldenrod', markersize=10, label='Snow pit'),
    Line2D([0], [0], marker='v', color='none', markerfacecolor='red',markeredgecolor='red', markersize=5, label='Magnaprobe'),
    
]
ax.legend(handles=custom_legend_elements, frameon=False, loc='lower left', bbox_to_anchor=(0.01, 0.01),handlelength=1.25)

ax.set_xlim(xlims[0] , xlims[1] )
ax.set_ylim(ylims[0], ylims[1])


def label_utm_grid():
    ''' Warning: should only use with small area UTM maps '''
    ax = plt.gca()    
    for val,label in zip(ax.get_xticks(), ax.get_xticklabels()):
        label.set_text(str(val))
        label.set_position((val,0))  

    for val,label in zip(ax.get_yticks(), ax.get_yticklabels()):   
        label.set_text(str(val))
        label.set_position((0,val))  

    plt.tick_params(bottom=True,top=True,left=True,right=True,
            labelbottom=True,labeltop=False,labelleft=True,labelright=False)

    ax.xaxis.set_visible(True)
    ax.yaxis.set_visible(True)
    ax.grid(ls=':', lw=.25)
    
# ax.gridlines(draw_labels=True, ls='')
label_utm_grid()


lin,  = ax.plot(RADARs['26_04_2024_5'].log_UTM_x[1000:1750], RADARs['26_04_2024_5'].log_UTM_y[1000:1750], lw=10,
           zorder=0,
           alpha=.7,
           transform=ccrs.UTM(RADARs['26_04_2024_5'].utm_zone),
           color='limegreen',
           solid_capstyle='round'
            )


In [ ]:
plt.savefig('./Figures_paper/export/fig04_p1.eps', dpi=600)

In [ ]:
%matplotlib inline

fig, ax  = plt.subplots(2, 1, figsize=(10, 4.5), dpi=150, height_ratios=[1.5, 1], constrained_layout=True, sharex=True)

ax[0].set_ylim(320,50)

ax[1].spines['top'].set_visible(False)
ax[1].spines['right'].set_visible(False)
ax[1].grid(axis='y', color='grey', ls=":")
X, Y = np.meshgrid(RADARs['26_04_2024_5'].dist[1000:1750], RADARs['26_04_2024_5'].range_snow[:, 0])


ax[0].pcolormesh(X, Y, Quickboost(RADARs['26_04_2024_5'].rx_rpca[:, 1000:1750], 0), cmap=cmr.neutral)

ax[0].plot(RADARs['26_04_2024_5'].dist[1000:1750], [RADARs['26_04_2024_5'].range_snow[:, 0][i] for i in RADARs['26_04_2024_5'].PF_top_interface[1000:1750]], color='deepskyblue', linewidth=0.5)
ax[0].plot(RADARs['26_04_2024_5'].dist[1000:1750], [RADARs['26_04_2024_5'].range_snow[:, 0][i] for i in RADARs['26_04_2024_5'].PF_bottom_interface[1000:1750]], color='magenta', linewidth=0.5)


for p in RADARs['26_04_2024_5'].PF_internal_layers['paths']:
    py, px = list(zip(*p))
    px = np.array(px)
    py = np.array(py)
    if np.any((px > 1000) & (px < 1750)):
        ax[0].plot(np.array(RADARs['26_04_2024_5'].dist)[px], [RADARs['26_04_2024_5'].range_snow[:, 0][i + RADARs['26_04_2024_5'].PF_top_interface[j]] for i,j in zip(py, px)],
                   color='goldenrod', linewidth=0.5,
                   alpha=.7
                   )

ax[1].set_xlabel('Distance [m]')
ax[0].set_ylabel("Range in snow [cm]\n$\\varepsilon'_{ds} = 1.73$")
ax[1].set_ylabel("Snow depth\n[m]")


uwibass = RADARs['26_04_2024_5']
ax[1].fill_between(RADARs['26_04_2024_5'].dist[1000:1750],
                   RADARs['26_04_2024_5'].PF_snow_depth[1000:1750] + RADARs['26_04_2024_5'].PF_total_uncertainty[1000:1750],
                   RADARs['26_04_2024_5'].PF_snow_depth[1000:1750] - RADARs['26_04_2024_5'].PF_total_uncertainty[1000:1750],
                   color='black', label="$h_{snow} \pm \\varepsilon_{total}$", alpha=.5)

ax[1].fill_between(RADARs['26_04_2024_5'].dist[1000:1750],
                   RADARs['26_04_2024_5'].PF_snow_depth[1000:1750] + RADARs['26_04_2024_5'].PF_snow_uncertainty[1000:1750],
                   RADARs['26_04_2024_5'].PF_snow_depth[1000:1750] - RADARs['26_04_2024_5'].PF_snow_uncertainty[1000:1750],
                   color='dimgrey', label="$h_{snow} \pm \\varepsilon_{snow}$", alpha=.5)

ax[1].fill_between(RADARs['26_04_2024_5'].dist[1000:1750],
                   RADARs['26_04_2024_5'].PF_snow_depth[1000:1750] + RADARs['26_04_2024_5'].PF_radar_uncertainty[1000:1750],
                   RADARs['26_04_2024_5'].PF_snow_depth[1000:1750] - RADARs['26_04_2024_5'].PF_radar_uncertainty[1000:1750],
                   color='lightgray', label="$h_{snow} \pm \\varepsilon_{radar}$", alpha=1)

ax[1].legend(loc='upper left', ncol=3, frameon=False, bbox_to_anchor=(0.05, 1.05), columnspacing=.9, fontsize=12)

ax[0].set_xlim(RADARs['26_04_2024_5'].dist[1000], RADARs['26_04_2024_5'].dist[1750])


In [ ]:
plt.savefig('./Figures_paper/export/fig04_p2.eps', dpi=600)
